In [1]:
#

In [ ]:
teste()

Combinações únicas:
[(1, 3, 2, 4), (2, 1, 3, 4), (2, 1, 4, 3), (3, 2, 1, 4), (3, 2, 4, 1), (4, 1, 3, 2), (4, 2, 1, 3), (4, 3, 2, 1)]

Quantidade de combinações únicas:
8

Frequência de cada combinação:
PreFormAnswers
(4, 2, 1, 3)    3
(3, 2, 4, 1)    3
(1, 3, 2, 4)    3
(3, 2, 1, 4)    2
(4, 3, 2, 1)    1
(4, 1, 3, 2)    1
(2, 1, 3, 4)    1
(2, 1, 4, 3)    1
Name: count, dtype: int64

Frequência no dungeon_frame final:
PreFormAnswers
[1, 3, 2, 4]    26
[4, 1, 3, 2]    25
[2, 1, 3, 4]    24
[4, 2, 1, 3]    18
[3, 2, 1, 4]    11
[3, 2, 4, 1]     9
[2, 1, 4, 3]     5
[4, 3, 2, 1]     1
Name: count, dtype: int64

Total de colunas: 57

  1. CompletedAchievementQuests
  2. CompletedCreativityQuests
  3. CompletedDamageQuests
  4. CompletedExchangeQuests
  5. CompletedExploreQuests
  6. CompletedGatherQuests
  7. CompletedGiveQuests
  8. CompletedGoToQuests
  9. CompletedImmersionQuests
 10. CompletedKillQuests
 11. CompletedListenQuests
 12. CompletedMasteryQuests
 13. CompletedQuests
 14. C

In [40]:
import random

import numpy as np
import json
import seaborn as sns
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.tree import DecisionTreeClassifier
from sklearn import svm
from sklearn.linear_model import SGDClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import cross_val_score
import os
import time
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, cohen_kappa_score
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split


from sklearn.metrics import mean_squared_error, mean_absolute_error
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# -----------------------------
# GRUPOS DE MÉTODOS
# -----------------------------
ITEM_PROFILE_METHODS = {1,2,5,6}
USER_PROFILE_METHODS = {3,4,7,8}
STRUCTURE_ONLY_METHODS = {9,10}
STRUCTURE_PERF_METHODS = {11,12,13}
HYBRID_PREFORM_METHODS = {14,15}
HYBRID_LEVEL_METHODS = {16,17}
HYBRID_STRUCTURE_METHODS = {18,19}
UNNAMED = {20,21}

# ------------------------------------------------- #
# 1- Entrada -> dungeons com perfis, ex: A=4, C=2, M=1, I=3; Similiaridade cosseno; Filtragem baseada em item (nota -> PostFormAnswers)
# *[OK]
# 2- Entrada -> dungeons com perfis, ex: A=4, C=2, M=1, I=3; Correlação de Pearson; Filtragem baseada em item (nota -> PostFormAnswers)
# *[OK]
# 3- Entrada -> dungeons com perfis, ex: A=4, C=2, M=1, I=3; Similiaridade cosseno; Filtragem baseada no usuário (comportamento -> SuccessRate, TotalLostHealth_norm, EnemyKillRate, MapCompletion)
# *[OK]
# 4- Entrada -> dungeons com perfis, ex: A=4, C=2, M=1, I=3; Correlação de Pearson; Filtragem baseada no usuário (comportamento -> SuccessRate, TotalLostHealth_norm, EnemyKillRate, MapCompletion)
# *[OK]
# 14 - Hibrido cosseno
# 15 - Hibrido pearson
# ------------------------------------------------- #

# 5- Entrada -> nome das dungeons, ex: M:20T:30G:10E:40; Similiaridade cosseno; Filtragem baseada em item (nota -> PostFormAnswers)
# *[OK]
# 6- Entrada -> nome das dungeons, ex: M:20T:30G:10E:40; Correlação de Pearson; Filtragem baseada em item (nota -> PostFormAnswers)
# *[OK]
# 7- Entrada -> nome das dungeons, ex: M:20T:30G:10E:40; Similiaridade cosseno; Filtragem baseada no usuário (comportamento -> SuccessRate, TotalLostHealth_norm, EnemyKillRate, MapCompletion)
# *[OK]
# 8- Entrada -> nome das dungeons, ex: M:20T:30G:10E:40; Correlação de Pearson; Filtragem baseada no usuário (comportamento -> SuccessRate, TotalLostHealth_norm, EnemyKillRate, MapCompletion)
# *[OK]
# 16 - Hibrido cosseno
# 17 - Hibrido pearson
# ------------------------------------------------- #

# 9- Entrada -> granular, ex: Dugeon com esses dados: 20 salas, 4 chaves-fechaduras, 1.5 de linearidad; Similiaridade cosseno; Filtragem baseada em item (nota -> PostFormAnswers)
# *
# 10- Entrada -> granular, ex: Dugeon com esses dados: 20 salas, 4 chaves-fechaduras, 1.5 de linearidad; Correlação de Pearson; Filtragem baseada em item (nota -> PostFormAnswers)
# *
# 11 - Entrada -> granular, ex: Dugeon com esses dados: 20 salas, 4 chaves-fechaduras, 1.5 de linearidad; Similiaridade cosseno; Filtragem baseada no usuário (comportamento -> SuccessRate, TotalLostHealth_norm, EnemyKillRate, MapCompletion)
# *
# 12 - Entrada -> granular, ex: Dugeon com esses dados: 20 salas, 4 chaves-fechaduras, 1.5 de linearidad; Correlação de Pearson; Filtragem baseada no usuário (comportamento -> SuccessRate, TotalLostHealth_norm, EnemyKillRate, MapCompletion)
# *[OK]
# 18 - Hibrido cosseno
# 19 - Hibrido pearson
# ------------------------------------------------- #
# 13 - multiclasse
# ------------------------------------------------- #


def initialize_data_frame():
    pd.set_option('display.max_columns', None)
    pd.set_option('display.width', 2000)
    sns.set(rc={'figure.figsize':(19.20, 10.80)})


def get_dungeon_data_frame():
    with open('backup.json') as data_file:
        json_data = json.load(data_file)
    return pd.DataFrame(json_data['__collections__']['dungeons']).T


def get_user_data_frame():
    with open('backup.json') as data_file:
        json_data = json.load(data_file)
    return pd.DataFrame(json_data['__collections__']['users']).T


def drop_invalid_rows(_data_frame):
    _data_frame.drop(_data_frame[_data_frame['UniqueRoomsEntered'] == 0].index, inplace=True)
    return _data_frame


def drop_unused_columns(_data_frame):
    _data_frame.drop('__collections__', inplace=True, axis=1)
    _data_frame = _data_frame.drop(list(_data_frame.filter(regex='^Total.*Quests')), axis=1)
    _data_frame.drop('CompletedQuests', inplace=True, axis=1)
    _data_frame.drop('CompletedKillQuests', inplace=True, axis=1)
    _data_frame.rename(columns={'CompletedDamageQuests': 'CompletedKillQuests'}, inplace=True)
    _data_frame.drop('TreasuresCollected', inplace=True, axis=1)
    _data_frame.drop('TotalNpcs', inplace=True, axis=1)
    _data_frame.drop('WeaponName', inplace=True, axis=1)
    _data_frame.drop('HeatMap', inplace=True, axis=1)
    return _data_frame


def get_percentages(_data_frame):
    _data_frame['LockUsage'] = _data_frame['LocksOpened'].div(_data_frame['TotalLocks'].values)*100
    _data_frame['KeyCollection'] = _data_frame['KeysCollected'].div(_data_frame['TotalKeys'].values)*100
    _data_frame['MapCompletion'] = _data_frame['UniqueRoomsEntered'].div(_data_frame['TotalRooms'].values)*100
    _data_frame['RoomRevisitRate'] = _data_frame['RoomsEntered'].div(_data_frame['TotalRooms'].values)*100
    _data_frame['EnemyKillRate'] = _data_frame['EnemiesKilled'].div(_data_frame['TotalEnemies'].values)*100
    _data_frame['TotalAttempts'] = _data_frame['TotalWins'] + _data_frame['TotalDeaths'].values
    _data_frame['TotalAttempts'] = _data_frame['TotalAttempts'].astype(float)
    _data_frame['TotalWins'] = _data_frame['TotalWins'].astype(float)
    _data_frame['Success Rate'] = _data_frame['TotalAttempts'].div(_data_frame['TotalWins'].values)*100
    _data_frame.replace([np.inf, -np.inf], 0, inplace=True)
    return _data_frame


def drop_processed_columns(_data_frame):
    _data_frame.drop('LocksOpened', inplace=True, axis=1)
    _data_frame.drop('TotalLocks', inplace=True, axis=1)
    _data_frame.drop('KeysCollected', inplace=True, axis=1)
    _data_frame.drop('TotalKeys', inplace=True, axis=1)
    _data_frame.drop('EnemiesKilled', inplace=True, axis=1)
    _data_frame.drop('TotalEnemies', inplace=True, axis=1)
    _data_frame.drop('TotalAttempts', inplace=True, axis=1)
    _data_frame.drop('TotalWins', inplace=True, axis=1)
    _data_frame.drop('TotalDeaths', inplace=True, axis=1)
    _data_frame.drop('NpcsInteracted', inplace=True, axis=1)
    _data_frame.drop('RoomsEntered', inplace=True, axis=1)
    _data_frame.drop('TotalRooms', inplace=True, axis=1)
    _data_frame.drop('UniqueRoomsEntered', inplace=True, axis=1)
    _data_frame.drop('TotalTreasure', inplace=True, axis=1)
    return _data_frame


def split_preform_answers(_data_frame):
    _data_frame[['A', 'C', 'I', 'M']] = _data_frame['PreFormAnswers'].str.split(',', expand=True)
    _data_frame['A'] = _data_frame['A'].apply(lambda x: ''.join((char for char in x if char.isdigit())))
    _data_frame['C'] = _data_frame['C'].apply(lambda x: ''.join((char for char in x if char.isdigit())))
    _data_frame['I'] = _data_frame['I'].apply(lambda x: ''.join((char for char in x if char.isdigit())))
    _data_frame['M'] = _data_frame['M'].apply(lambda x: ''.join((char for char in x if char.isdigit())))
    return _data_frame


def augment_dataframe_with_scaling(df, y, n, scale_factor):
    """
    Augments the DataFrame by adding `n` new rows, scaling numeric columns by a given factor.
    The new rows inherit the same 'y' values as their corresponding sampled rows.

    Parameters:
        df (pd.DataFrame): Original DataFrame with numeric columns and a 'y' column.
        n (int): Number of new rows to generate.
        scale_factor (float): The factor by which numeric columns will be scaled.

    Returns:
        pd.DataFrame: DataFrame with the original and new rows.
    """
    # Identify numeric columns, excluding 'y'
    numeric_columns = df.select_dtypes(include=['number']).columns.difference(y)
    # Randomly sample `n` rows from the original DataFrame
    sampled_rows = df.sample(n=n, replace=True).reset_index(drop=True)

    # Scale numeric columns
    for col in numeric_columns:
        sampled_rows[col] *= scale_factor

    # Append the scaled rows to the original DataFrame
    augmented_df = pd.concat([df, sampled_rows], ignore_index=True)

    return augmented_df


def augment_data_random(_df, y, n):
    """
    Augments the DataFrame by adding `n` new rows.
    - Adds noise to numerical columns using a normal distribution.
    - Retains the same 'y' values for the newly added rows as their corresponding sampled rows.

    Parameters:
        _df (pd.DataFrame): Original DataFrame with numeric columns and a 'y' column.
        y (list): the column(s) that will have their value retained
        n (int): Number of new rows to generate.

    Returns:
        pd.DataFrame: DataFrame with the original and new rows.
    """
    # Identify float and integer columns, excluding 'y'
    float_columns = _df.select_dtypes(include=['float']).columns.difference(y)
    int_columns = _df.select_dtypes(include=['integer']).columns.difference(y)

    # Randomly sample `n` rows from the original DataFrame
    sampled_rows = _df.sample(n=n, replace=True).reset_index(drop=True)

    # Add noise to float columns
    for col in float_columns:
        if random.randint(1, 100) > 50:
            std = _df[col].std()
            noise = np.random.normal(loc=0, scale=std, size=n)
            sampled_rows[col] += noise

    for col in int_columns:
        if random.randint(1, 100) > 50:
            std = _df[col].std()
            noise = np.random.normal(loc=0, scale=std, size=n)
            sampled_rows[col] += noise

    # Ensure integer columns remain integers
    sampled_rows[int_columns] = sampled_rows[int_columns].round().astype(int)

    # Append the sampled (augmented) rows to the original DataFrame
    augmented_df = pd.concat([_df, sampled_rows], ignore_index=True)
    return augmented_df

def recommend_from_other_user(target_user, other_user, user_item_matrix):

    print(f"\nRecomendando para usuário {target_user} com base no usuário {other_user}")

    # Itens que target_user já jogou
    target_items = user_item_matrix.loc[target_user]

    # Itens que other_user já jogou
    other_items = user_item_matrix.loc[other_user]

    # Pegar apenas itens que:
    # - other_user jogou (rating > 0)
    # - target_user NÃO jogou (rating == 0)
    candidate_items = other_items[
        (other_items > 0) & (target_items == 0)
    ]

    if len(candidate_items) == 0:
        print("Nenhum item novo para recomendar.")
        return

    # Ordenar pelos melhores ratings do other_user
    recommendations = candidate_items.sort_values(ascending=False)

    print("\nItens recomendados:")
    print(recommendations.head(5))


# ---------------- RECOMENDAÇÃO USER-BASED EXPLICÁVEL ---------------- #

def recommend_user_based_explain(target_user, user_item_matrix, user_similarity_df, top_n=5):

    print(f"\nRecomendações para usuário {target_user}")

    # Similaridade do usuário alvo com todos os outros
    similarities = user_similarity_df.loc[target_user].drop(target_user)
    # Pegar itens que o usuário ainda NÃO jogou
    user_ratings = user_item_matrix.loc[target_user]
    items_not_played = user_ratings[user_ratings == 0].index

    predictions = {}
    explanations = {}

    for item in items_not_played:
        # Usuários que avaliaram esse item
        users_who_rated = user_item_matrix[user_item_matrix[item] > 0].index

        numerator = 0
        denominator = 0
        contributors = []

        for other_user in users_who_rated:

            sim = similarities.get(other_user, 0)

            if sim > 0:# só similaridade positiva
                rating = user_item_matrix.loc[other_user, item]
                contribution = sim * rating

                numerator += contribution
                denominator += abs(sim)

                contributors.append({
                    "user": other_user,
                    "similarity": sim,
                    "rating": rating,
                    "contribution": contribution
                })

        if denominator > 0:
            predicted_rating = numerator / denominator
            predictions[item] = predicted_rating
            explanations[item] = contributors

    if len(predictions) == 0:
        print("Nenhum item possível para recomendar.")
        return

    recommendations = (
        pd.Series(predictions)
        .sort_values(ascending=False)
        .head(top_n)
    )

    '''print("\nItens recomendados:")
    print(recommendations)'''

    # Mostrar explicação detalhada
    for item in recommendations.index:
        print("\n---------------------------------------")
        print(f"Explicação para {item}")
        print("---------------------------------------")

        contribs = sorted(
            explanations[item],
            key=lambda x: x["contribution"],
            reverse=True
        )

        for c in contribs[:5]:  # top 5 contribuidores
            print(
                f"Usuário {c['user']} | "
                f"similaridade={round(c['similarity'],3)} | "
                f"nota={round(c['rating'],2)} | "
                f"contribuição={round(c['contribution'],3)}"
            )
# Recommend items similar to 'Item1'
def recommend_similar_items(item, similarity_df, top_n=3):
  return similarity_df[item].sort_values(ascending=False)[1:top_n+1]

def print_user_ratings(user_item_matrix, user_similarity_df, user1_idx=0, user2_idx=1):
    # seleciona usuários
    u1 = user_similarity_df.index[user1_idx]
    u2 = user_similarity_df.index[user2_idx]

    print("Notas do usuário 1:")
    print(user_item_matrix.loc[u1].dropna())

    print("\nNotas do usuário 2:")
    print(user_item_matrix.loc[u2].dropna())

def analyze_user_activity(user_item_matrix):
    # Contar quantos níveis cada usuário jogou
    levels_per_user = user_item_matrix.notna().sum(axis=1)

    # Filtrar usuários que jogaram mais de 1 nível
    users_more_than_one = levels_per_user[levels_per_user > 1]

    print("Usuários que jogaram mais de um nível:")
    print(users_more_than_one.head())

    # Pegar o primeiro deles
    if len(users_more_than_one) > 0:
        u = users_more_than_one.index[0]

        print("\nUsuário selecionado:", u)
        print("\nQuantidade de níveis jogados:", levels_per_user.loc[u])

        print("\nNotas desse usuário:")
        print(user_item_matrix.loc[u].dropna())
    else:
        print("Nenhum usuário jogou mais de um nível.")

    # Distribuição geral
    print("\nDistribuição:")
    print(levels_per_user.value_counts().sort_index())

def build_quest_df(df):

    quest_cols = [col for col in df.columns if "Completed" in col and "Quests" in col]

    quest_df = df[['PlayerId'] + quest_cols].copy()

    # transformar em formato longo
    quest_df = quest_df.melt(
        id_vars='PlayerId',
        var_name='Item',
        value_name='Rating'
    )

    # remover zeros
    quest_df = quest_df[quest_df['Rating'] > 0]

    quest_df.columns = ['User', 'Item', 'Rating']

    return quest_df

def build_enemy_df(df):

    df = df.copy()

    # dificuldade simples (você pode melhorar depois)
    df['EnemyDifficulty'] = pd.cut(
        df['TotalEnemies'],
        bins=3,
        labels=['Easy', 'Medium', 'Hard']
    )

    enemy_df = df[['PlayerId', 'EnemyDifficulty', 'EnemiesKilled']].copy()

    enemy_df.columns = ['User', 'Item', 'Rating']

    return enemy_df

def build_struct_df(df):

    df = df.copy()

    df['Item'] = (
        "R" + df['TotalRooms'].astype(str) +
        "_K" + df['TotalKeys'].astype(str) +
        "_RR" + (df['RoomsEntered'] - df['UniqueRoomsEntered']).astype(str)
    )

    '''df['Rating'] = df['PostFormAnswers'].apply(
        lambda x: sum(x)/len(x) if isinstance(x, list) else x
    )'''
    df['Rating'] = df['PostFormAnswers'].apply(
        lambda x: sum(x)/len(x) if isinstance(x,list) else (0 if x is None else x))


    struct_df = df[['PlayerId', 'Item', 'Rating']]
    struct_df.columns = ['User', 'Item', 'Rating']

    return struct_df

def multi_view_recommendation(df, target_user):

    quest_df  = build_quest_df(df)
    enemy_df  = build_enemy_df(df)
    struct_df = build_struct_df(df)

    # gerar matrizes
    def build_matrix(d):
        return d.pivot_table(index='User', columns='Item', values='Rating', fill_value=0)

    quest_matrix  = build_matrix(quest_df)
    enemy_matrix  = build_matrix(enemy_df)
    struct_matrix = build_matrix(struct_df)

    # similaridade
    def sim(m):
        return pd.DataFrame(
            cosine_similarity(m),
            index=m.index,
            columns=m.index
        )

    quest_sim  = sim(quest_matrix)
    enemy_sim  = sim(enemy_matrix)
    struct_sim = sim(struct_matrix)

    print("\n===== QUEST =====")
    recommend_user_based_explain(target_user, quest_matrix, quest_sim)

    print("\n===== ENEMY =====")
    recommend_user_based_explain(target_user, enemy_matrix, enemy_sim)

    print("\n===== STRUCT =====")
    recommend_user_based_explain(target_user, struct_matrix, struct_sim)


def predict_item_based(user, item, user_item_matrix, item_similarity_df, k=20):
    if item not in item_similarity_df.columns or user not in user_item_matrix.index:
        return 0  # ou np.nan
    similar_items = item_similarity_df[item]
    user_ratings = user_item_matrix.loc[user]
    # 👉 Filtrar itens não avaliados pelo usuário
    mask = user_ratings != 0
    similar_items = similar_items[mask]; user_ratings = user_ratings[mask]
    if similar_items.empty:
        return 0
    # (Opcional) usar apenas os top-K vizinhos
    similar_items = similar_items.sort_values(ascending=False).head(k)
    user_ratings = user_ratings.loc[similar_items.index]
    numerator = (similar_items * user_ratings).sum()
    denominator = similar_items.abs().sum()
    return numerator/denominator if denominator != 0 else 0

def predict_user_based(user, item, user_item_matrix, user_similarity_df, k=20):
    if user not in user_similarity_df.index or item not in user_item_matrix.columns:
        return 0
    similar_users = user_similarity_df.loc[user].drop(user)
    item_ratings = user_item_matrix[item]
    mask = item_ratings != 0
    similar_users = similar_users[mask]; item_ratings = item_ratings[mask]
    if similar_users.empty:
        return 0
    similar_users = similar_users.sort_values(ascending=False).head(k)
    item_ratings = item_ratings.loc[similar_users.index]
    numerator = (similar_users * item_ratings).sum()
    denominator = similar_users.abs().sum()
    return numerator/denominator if denominator != 0 else 0


def predict_hybrid(user, item,
                   user_item_matrix,
                   item_similarity_df,
                   user_similarity_df,
                   alpha=0.5):

    score_item = predict_item_based(
        user, item,
        user_item_matrix,
        item_similarity_df
    )

    score_user = predict_user_based(
        user, item,
        user_item_matrix,
        user_similarity_df
    )

    return alpha * score_item + (1 - alpha) * score_user

def recommend_hybrid(user,
                     user_item_matrix,
                     item_similarity_df,
                     user_similarity_df,
                     top_n=5,
                     alpha=0.5):

    items = user_item_matrix.columns
    predictions = {}

    for item in items:
        if user_item_matrix.loc[user, item] == 0:  # não avaliou
            predictions[item] = predict_hybrid(
                user,
                item,
                user_item_matrix,
                item_similarity_df,
                user_similarity_df,
                alpha
            )

    ranked = sorted(predictions.items(),
                    key=lambda x: x[1],
                    reverse=True)

    return ranked[:top_n]

def mean_post_forms(x):
    return sum(x)/len(x) if isinstance(x, list) else x

def minmax(series):
    return (series - series.min()) / (series.max() - series.min() + 1e-9)

def compute_performance_rating(df):
    df['TotalLostHealth_norm'] = minmax(df['TotalLostHealth'])

    return (
        df['SuccessRate']
        - df['TotalLostHealth_norm']
        + df['EnemyKillRate']
        + df['MapCompletion']
    )

def build_item_from_structure(df):
    df['TotalRooms'] = df['TotalRooms'].replace(0, 1)

    return (
        "R" + df['TotalRooms'].astype(str) +
        "_K" + df['TotalKeys'].astype(str) +
        "_RR" + (df['RoomsEntered'] - df['UniqueRoomsEntered']).astype(str)
    )

def compute_hybrid_rating(df, alpha=0.5):
    df_valid = df['PostFormAnswers'].apply(mean_post_forms)
    df_comp = compute_performance_rating(df)

    return alpha * df_comp + (1 - alpha) * df_valid

def build_collab_df(_df, method):

    '''if method in ITEM_PROFILE_METHODS:
      if method in {1,2}:
          item_col = 'PreFormAnswers'
          cols = ['PlayerId', item_col, 'PostFormAnswers']
          df = _df[cols].copy()
      else:  # methods 5,6
          df = _df.copy()  # ← trabalhar na cópia
          df['Item'] = pd.cut(
              df['TotalTreasure'],
              bins=[0, 5, 10, 15, 20, 100],
              labels=['T1', 'T2', 'T3', 'T4', 'T5']
          )
          df = df[['PlayerId', 'Item', 'PostFormAnswers']].copy()

      df.columns = ['User', 'Item', 'Rating']
      df['Rating'] = df['Rating'].apply(mean_post_forms)
      df['Rating'] = minmax(df['Rating'])
      df['Item'] = df['Item'].astype(str)

      return df[['User', 'Item', 'Rating']]'''

    # ----------------------------
    # ITEM PROFILE
    # ----------------------------
    if method in ITEM_PROFILE_METHODS:

        if method in {1,2}:
            item_col = 'PreFormAnswers'
        else:
            item_col = 'LevelName'

        cols = ['PlayerId', item_col, 'PostFormAnswers']

        df = _df[cols].copy()
        df.columns = ['User', 'Item', 'Rating']
        df['Rating'] = df['Rating'].apply(mean_post_forms)
        df['Rating'] = minmax(df['Rating'])
        return df[['User','Item','Rating']]
    # ----------------------------
    # USER PROFILE (performance)
    # ----------------------------
    elif method in USER_PROFILE_METHODS:

        if method in {3,4}:
            item_col = 'PreFormAnswers'
        else:
            item_col = 'LevelName'

        cols = ['PlayerId', item_col, 'Success Rate',
                'TotalLostHealth', 'EnemyKillRate', 'MapCompletion']

        df = _df[cols].copy()
        df.columns = ['User', 'Item', 'SuccessRate',
                      'TotalLostHealth', 'EnemyKillRate', 'MapCompletion']


        df['Rating'] = compute_performance_rating(df)
        df = df[['User','Item','Rating']]
        df['Rating'] = minmax(df['Rating'])
        return df[['User','Item','Rating']]

    # ----------------------------
    # STRUCTURE BASED
    # ----------------------------
    elif method in STRUCTURE_ONLY_METHODS.union(STRUCTURE_PERF_METHODS):

        cols = ['PlayerId','TotalRooms','TotalKeys',
                'RoomsEntered','UniqueRoomsEntered']

        if method in STRUCTURE_ONLY_METHODS:
            cols.append('PostFormAnswers')
        else:
            cols += ['Success Rate','TotalLostHealth',
                     'EnemyKillRate','MapCompletion']

        df = _df[cols].copy()
        if method in STRUCTURE_ONLY_METHODS:
          df.columns = ['User', 'TotalRooms', 'TotalKeys', 'RoomsEntered', 'UniqueRoomsEntered', 'Rating']
        else:
            df.columns = ['User', 'TotalRooms', 'TotalKeys', 'RoomsEntered', 'UniqueRoomsEntered', 'SuccessRate',
                        'TotalLostHealth',
                        'EnemyKillRate',
                        'MapCompletion']

        df['Item'] = build_item_from_structure(df)

        if method in STRUCTURE_ONLY_METHODS:
            df['Rating'] = df['Rating'].apply(mean_post_forms)
        else:

            df['Rating'] = compute_performance_rating(df)

        df['Rating'] = minmax(df['Rating'])
        df = df[['User','Item','Rating']]
        return df[['User','Item','Rating']]

    # --------------------------------------------------
    # HYBRID PREFORM (14,15)
    # --------------------------------------------------
    elif method in HYBRID_PREFORM_METHODS:

        cols = ['PlayerId','PreFormAnswers','PostFormAnswers',
                'Success Rate','TotalLostHealth',
                'EnemyKillRate','MapCompletion']

        df = _df[cols].copy()
        df.columns = ['User','Item','PostFormAnswers',
                      'SuccessRate','TotalLostHealth',
                      'EnemyKillRate','MapCompletion']

        df['Rating'] = compute_hybrid_rating(df)
        df['Rating'] = minmax(df['Rating'])
        return df[['User','Item','Rating']]


    # --------------------------------------------------
    # HYBRID LEVEL (16,17)
    # --------------------------------------------------
    elif method in HYBRID_LEVEL_METHODS:

        cols = ['PlayerId','LevelName','PostFormAnswers',
                'Success Rate','TotalLostHealth',
                'EnemyKillRate','MapCompletion']

        df = _df[cols].copy()
        df.columns = ['User','Item','PostFormAnswers',
                      'SuccessRate','TotalLostHealth',
                      'EnemyKillRate','MapCompletion']

        df['Rating'] = compute_hybrid_rating(df)
        df['Rating'] = minmax(df['Rating'])
        return df[['User','Item','Rating']]


    # --------------------------------------------------
    # HYBRID STRUCTURE (18,19)
    # --------------------------------------------------
    elif method in HYBRID_STRUCTURE_METHODS:

        cols = ['PlayerId','TotalRooms','TotalKeys',
                'RoomsEntered','UniqueRoomsEntered',
                'PostFormAnswers','Success Rate',
                'TotalLostHealth','EnemyKillRate','MapCompletion']

        df = _df[cols].copy()
        df.columns = ['User','TotalRooms','TotalKeys',
                      'RoomsEntered','UniqueRoomsEntered',
                      'PostFormAnswers','SuccessRate',
                      'TotalLostHealth','EnemyKillRate','MapCompletion']

        df['Item'] = build_item_from_structure(df)
        df['Rating'] = compute_hybrid_rating(df)
        df['Rating'] = minmax(df['Rating'])
        return df[['User','Item','Rating']]
    elif method in UNNAMED:

      cols = [
          'PlayerId',

          # combate
          'EnemiesKilled',
          'MaxCombo',

          # quests combate
          'CompletedKillQuests',
          'CompletedDamageQuests'
      ]

      if method == 20:
            cols.append('PostFormAnswers')
      else:
            cols += ['Success Rate','TotalLostHealth','MapCompletion','EnemyKillRate']


      df = _df[cols].copy()

      # -----------------------------------
      # RENOMEAR
      # -----------------------------------

      if method == 20:

        df.columns = [
            'User',
            'EnemiesKilled',
            'MaxCombo',
            'CompletedKillQuests',
            'CompletedDamageQuests',
            'Rating'
        ]

      else:

          df.columns = [
              'User',
              'EnemiesKilled',
              'MaxCombo',
              'CompletedKillQuests',
              'CompletedDamageQuests',
              'SuccessRate',
              'TotalLostHealth',
              'MapCompletion',
              'EnemyKillRate',
          ]

      # ===================================
      # COMBAT SCORE
      # ===================================

      df['CombatScore'] = (
          df['EnemiesKilled'] +
          df['MaxCombo'] +
          df['CompletedKillQuests'] +
          df['CompletedDamageQuests']
      ) / 4

      # ===================================
      # NORMALIZA
      # ===================================

      df['CombatScore'] = minmax(          df['CombatScore']      )

      # ===================================
      # BUCKET
      # ===================================

      def combat_bucket(x):

          if x < 0.33:
              return 'COMBAT_LOW'

          elif x < 0.66:
              return 'COMBAT_MEDIUM'

          return 'COMBAT_HIGH'

      # ===================================
      # ITEM
      # ===================================

      df['Item'] = df['CombatScore'].apply(
          combat_bucket
      )

      # ===================================
      # RATING
      # ===================================

      if method == 20:
            df['Rating'] = df['Rating'].apply(mean_post_forms)
      else:
            df['Rating'] = compute_performance_rating(df)


      # ===================================
      # NORMALIZA RATING
      # ===================================

      df['Rating'] = minmax(
          df['Rating']
      )

      # ===================================
      # FINAL
      # ===================================

      df = df[['User', 'Item', 'Rating']]
      return df

    else:
        raise ValueError(f"Método {method} inválido.")

    return df

def analyze_user_item_matrix(user_item_matrix):

    print("------------- VERIFICAÇÃO DOS DADOS -------------")
    print("Número de usuários:", user_item_matrix.shape[0])
    print("Número de itens:", user_item_matrix.shape[1])

    print("\nQuantidade média de avaliações por usuário:")
    print(user_item_matrix.count(axis=1).mean())

    print("\nQuantidade média de usuários por item:")
    print(user_item_matrix.count(axis=0).mean())

    print("\n------------- VARIÂNCIA ITEM POR ITEM -------------")
    print("Variância de cada item:")
    print(user_item_matrix.var())

def similiarity_pearson(user_item_matrix):
    # Pearson item-item
    item_similarity_pearson = user_item_matrix.corr(method='pearson')

    print("\nITEM POR ITEM (PEARSON):\n")
    print(item_similarity_pearson)
    item_similarity_df = item_similarity_pearson

    user_similarity_pearson = user_item_matrix.T.corr(method='pearson')
    user_similarity_pearson = user_similarity_pearson.fillna(0)

    print("\nUSUARIO POR USUARIO (PEARSON):\n")
    print(user_similarity_pearson)

    user_similarity_df  = user_similarity_pearson

    return item_similarity_df, user_similarity_df

def similiarity_cosine(user_item_matrix):
    # Compute item similarity using cosine similarity
    item_similarity = cosine_similarity(user_item_matrix.T)
    item_similarity_df = pd.DataFrame(item_similarity, index=user_item_matrix.columns, columns=user_item_matrix.columns)

    #centralizando
    user_item_centered = user_item_matrix.sub(user_item_matrix.mean(axis=1), axis=0)
    #print("\nconsseno cetnralizado:\n")
    #print(user_item_centered)

    # Similaridade entre itens
    # item por item
    item_similarity_centered = cosine_similarity(user_item_centered.T)
    item_similarity_centered_df = pd.DataFrame(item_similarity_centered, index=user_item_centered.columns, columns=user_item_centered.columns)

    #print("\nconsseno:\n")
    #print(item_similarity_df)

    #print("\nconsseno cetnralizado:\n")
    #print(item_similarity_centered)

    # usuario por usuário
    user_similarity = cosine_similarity(user_item_centered)
    user_similarity_df = pd.DataFrame( user_similarity, index=user_item_matrix.index, columns=user_item_matrix.index )
    #print("\nUSUARIO POR USUARIO:\n")
    #print(user_similarity_df.iloc[:50, :50])

    return item_similarity_df, user_similarity_df

def train_test_split_interactions(df, test_size=0.2):
    train, test = train_test_split(df, test_size=test_size, random_state=42)
    return train, test

def predict_all_users(user_item_matrix, similarity_matrix, method="user"):
    predictions = []

    for user in user_item_matrix.index:
        for item in user_item_matrix.columns:
            if user_item_matrix.loc[user, item] == 0:

                if method == "user":
                    pred = predict_user_based(user, item, user_item_matrix, similarity_matrix)
                else:
                    pred = predict_item_based(user, item, user_item_matrix, similarity_matrix)

                predictions.append((user, item, pred))

    return pd.DataFrame(predictions, columns=["User", "Item", "Prediction"])

def evaluate_rating_metrics(predictions_df, test_matrix):

    y_true = []
    y_pred = []

    for _, row in predictions_df.iterrows():
        user = row["User"]
        item = row["Item"]

        if user in test_matrix.index and item in test_matrix.columns:
            true_rating = test_matrix.loc[user, item]
            if true_rating != 0:
                y_true.append(true_rating)
                y_pred.append(row["Prediction"])

    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)

    return rmse, mae

def compute_rmse(test_df, prediction_col, rating_col):
    """
    Calcula RMSE ignorando:
        - valores NaN
        - casos sem predição
        - divisão por zero
    """

    # Manter apenas linhas com valor real e predito
    mask = (
        test_df[rating_col].notna() &
        test_df[prediction_col].notna()
    )

    y_true = test_df.loc[mask, rating_col].values
    y_pred = test_df.loc[mask, prediction_col].values

    if len(y_true) == 0:
        raise ValueError("Nenhuma predição válida para calcular RMSE.")

    rmse = np.sqrt(np.mean((y_true - y_pred) ** 2))

    return rmse

def precision_at_k(rec_df, test_df, k=5, threshold=1.0):
    precision_list = []

    for user in test_df["User"].unique():

        top_k = rec_df[rec_df["User"] == user].head(k)["Item"].values
        true_items = test_df[(test_df["User"] == user) & (test_df["Rating"] >= threshold)]["Item"].values

        if len(true_items) == 0:
            continue

        hits = len(set(top_k) & set(true_items))
        precision_list.append(hits / k)

    return np.mean(precision_list)
def recall_at_k(rec_df, test_df, k=5, threshold=1.0):
    recall_list = []

    for user in test_df["User"].unique():

        top_k = rec_df[rec_df["User"] == user].head(k)["Item"].values
        true_items = test_df[(test_df["User"] == user) & (test_df["Rating"] >= threshold)]["Item"].values

        if len(true_items) == 0:
            continue

        hits = len(set(top_k) & set(true_items))
        recall_list.append(hits / len(true_items))

    return np.mean(recall_list)
def hit_rate_at_k(rec_df, test_df, k=5):
    hits = 0
    users = test_df["User"].unique()

    for user in users:
        top_k = rec_df[rec_df["User"] == user].head(k)["Item"].values
        true_items = test_df[test_df["User"] == user]["Item"].values

        if len(set(top_k) & set(true_items)) > 0:
            hits += 1

    return hits / len(users)
def mrr(rec_df, test_df):
    rr_total = []

    for user in test_df["User"].unique():

        top_k = rec_df[rec_df["User"] == user].reset_index()

        true_items = set(test_df[test_df["User"] == user]["Item"])

        rank = 0

        for i, row in top_k.iterrows():
            if row["Item"] in true_items:
                rank = i + 1
                break

        if rank > 0:
            rr_total.append(1 / rank)

    return np.mean(rr_total)

def analyze_all_columns_cardinality(df):
    """
    Analisa cardinalidade de todas as colunas do DataFrame
    """
    results = []

    for col in df.columns:
        if df[col].dtype in ['int64', 'float64', 'object']:  # Apenas colunas que podem ser itens
            unique_vals = df[col].nunique()
            total_rows = len(df)
            ratio = unique_vals / total_rows

            results.append({
                'column': col,
                'dtype': df[col].dtype,
                'unique_values': unique_vals,
                'total_rows': total_rows,
                'ratio': ratio,
                'recommended_for_item': ratio < 0.01  # Menos de 1% de cardinalidade
            })

    # Ordenar por ratio (menor cardinalidade primeiro)
    results_df = pd.DataFrame(results).sort_values('ratio')

    print("\n" + "="*80)
    print("ANÁLISE DE CARDINALIDADE - TODAS AS COLUNAS")
    print("="*80)
    print(results_df.to_string(index=False))

    print("\n" + "="*80)
    print("COLUNAS RECOMENDADAS PARA ITEM-BASED CF (cardinalidade < 1%):")
    print("="*80)
    recommended = results_df[results_df['recommended_for_item']]
    if len(recommended) > 0:
        print(recommended[['column', 'unique_values', 'ratio']].to_string(index=False))
    else:
        print("NENHUMA coluna com baixa cardinalidade encontrada!")

    return results_df

def recommend_attributes(user_id, df, attribute_cols, top_n=5):
    """
    Recomenda valores para cada atributo separadamente
    """
    recommendations = {}

    for attr in attribute_cols:
        # Criar matriz usuário-atributo
        attr_matrix = df.pivot_table(
            index='PlayerId',
            columns=attr,
            values='Rating',  # ou 'PostFormAnswers' média
            fill_value=0
        )

        # Calcular similaridade
        from sklearn.metrics.pairwise import cosine_similarity

        if user_id in attr_matrix.index:
            user_sim = cosine_similarity(
                attr_matrix.loc[[user_id]],
                attr_matrix
            )[0]

            # Encontrar valores mais similares que o usuário ainda não viu
            user_values = set(df[df['PlayerId'] == user_id][attr].unique())
            all_values = set(df[attr].unique())
            new_values = all_values - user_values

            # Média ponderada pela similaridade
            scores = {}
            for val in new_values:
                users_with_val = df[df[attr] == val]['PlayerId'].unique()
                avg_sim = np.mean([user_sim[attr_matrix.index.get_loc(u)] for u in users_with_val if u in attr_matrix.index])
                scores[val] = avg_sim

            recommendations[attr] = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:top_n]

    return recommendations

def build_composite_item(df, method):
    """
    Cria um item que combina os atributos que você quer recomendar
    """
    df = df.copy()

    # Atributos que você quer recomendar
    components = []

    if 'TotalRooms' in df.columns:
        df['RoomBucket'] = pd.cut(
            df['TotalRooms'],
            bins=[0, 10, 15, 20, 25, 50],
            labels=['R_VS', 'R_S', 'R_M', 'R_L', 'R_VL']
        )
        components.append(df['RoomBucket'])

    if 'TotalKeys' in df.columns:
        df['KeyBucket'] = pd.cut(
            df['TotalKeys'],
            bins=[0, 2, 4, 6, 10],
            labels=['K_FEW', 'K_MED', 'K_MANY', 'K_LOT']
        )
        components.append(df['KeyBucket'])

    if 'TotalLocks' in df.columns:
        df['LockBucket'] = pd.cut(
            df['TotalLocks'],
            bins=[0, 2, 4, 6, 10],
            labels=['L_FEW', 'L_MED', 'L_MANY', 'L_LOT']
        )
        components.append(df['LockBucket'])

    if 'TotalEnemies' in df.columns:
        df['EnemyBucket'] = pd.cut(
            df['TotalEnemies'],
            bins=[0, 10, 20, 30, 50, 100],
            labels=['E_EASY', 'E_MED', 'E_HARD', 'E_VH', 'E_INSANE']
        )
        components.append(df['EnemyBucket'])

    # Combinar todos os atributos em um único item
    df['Item'] = "_".join([c.astype(str) for c in components])

    # Rating pode ser o pós-teste ou performance
    df['Rating'] = df['PostFormAnswers'].apply(mean_post_forms)

    return df[['PlayerId', 'Item', 'Rating']]

def recommend_by_feature_vector(user_id, df, feature_cols, n_recommendations=5):
    """
    Recomenda baseado em similaridade do vetor de características
    """
    from sklearn.preprocessing import MinMaxScaler
    from sklearn.metrics.pairwise import cosine_similarity

    # Selecionar características numéricas
    available_features = [f for f in feature_cols if f in df.columns]

    # Calcular vetor médio por dungeon/LevelName
    dungeon_features = df.groupby('LevelName')[available_features].mean()

    # Calcular vetor do usuário (média das dungeons que ele gostou)
    user_dungeons = df[df['PlayerId'] == user_id]['LevelName'].unique()
    if len(user_dungeons) == 0:
        return []

    user_vector = dungeon_features.loc[user_dungeons].mean()

    # Similaridade com todas as dungeons
    scaler = MinMaxScaler()
    features_scaled = scaler.fit_transform(dungeon_features)
    user_scaled = scaler.transform(user_vector.values.reshape(1, -1))

    similarities = cosine_similarity(user_scaled, features_scaled)[0]

    # Ordenar e recomendar
    results = pd.DataFrame({
        'Dungeon': dungeon_features.index,
        'Similarity': similarities
    }).sort_values('Similarity', ascending=False)

    # Filtrar dungeons que o usuário já jogou
    played = set(df[df['PlayerId'] == user_id]['LevelName'])
    recommendations = results[~results['Dungeon'].isin(played)].head(n_recommendations)

    # Mostrar os atributos recomendados
    for _, row in recommendations.iterrows():
        dungeon = row['Dungeon']
        attrs = dungeon_features.loc[dungeon]
        print(f"\nRecomendar: {dungeon}")
        print(f"  Características:")
        for col in available_features:
            print(f"    {col}: {attrs[col]:.1f}")

    return recommendations

def recommend_attributes_hybrid(user_id, df, method=20):
    """
    Recomenda atributos específicos (salas, chaves, inimigos, dificuldade)
    """
    from sklearn.preprocessing import MinMaxScaler
    from sklearn.metrics.pairwise import cosine_similarity

    # 1. Definir atributos que você quer recomendar
    attribute_groups = {
        'estrutura': ['TotalRooms', 'TotalKeys', 'TotalLocks'],
        'combate': ['TotalEnemies', 'EnemiesKilled', 'MaxCombo'],
        'exploracao': ['MapCompletion', 'UniqueRoomsEntered', 'RoomRevisitRate']
    }

    recommendations = {}

    for group_name, attrs in attribute_groups.items():
        available = [a for a in attrs if a in df.columns]
        if not available:
            continue

        # 2. Criar perfil do usuário
        user_data = df[df['PlayerId'] == user_id]
        if len(user_data) == 0:
            continue

        # 3. Para cada atributo, calcular valor preferido
        user_profile = {}
        for attr in available:
            # Média ponderada pela nota (PostFormAnswers)
            weights = user_data['PostFormAnswers'].apply(mean_post_forms)
            user_profile[attr] = np.average(user_data[attr], weights=weights)

        # 4. Comparar com todas as dungeons
        dungeon_avg = df.groupby('LevelName')[available].mean()

        # 5. Similaridade
        scaler = MinMaxScaler()
        user_vec = np.array([user_profile[a] for a in available]).reshape(1, -1)
        dungeon_vecs = scaler.fit_transform(dungeon_avg)
        user_scaled = scaler.transform(user_vec)

        similarities = cosine_similarity(user_scaled, dungeon_vecs)[0]

        # 6. Ordenar e recomendar
        results = pd.DataFrame({
            'Dungeon': dungeon_avg.index,
            'Similarity': similarities
        }).sort_values('Similarity', ascending=False)

        # Filtrar já jogadas
        played = set(user_data['LevelName'].unique())
        recommendations[group_name] = results[~results['Dungeon'].isin(played)].head(3)

        # 7. Mostrar recomendações com atributos
        print(f"\n{'='*50}")
        print(f"RECOMENDAÇÕES BASEADAS EM {group_name.upper()}")
        print(f"{'='*50}")

        for _, row in recommendations[group_name].iterrows():
            dungeon = row['Dungeon']
            attrs = dungeon_avg.loc[dungeon]
            print(f"\n📌 {dungeon} (similaridade: {row['Similarity']:.3f})")
            for a in available:
                print(f"   {a}: {attrs[a]:.1f}")

    return recommendations

# Uso no seu código
def main_with_attribute_recommendation(method=20):
    initialize_data_frame()
    dungeon_frame = get_dungeon_data_frame()
    user_frame = get_user_data_frame()
    dungeon_frame = pd.merge(dungeon_frame, user_frame[['PlayerId', 'PreFormAnswers']], on=['PlayerId'], how='left')
    dungeon_frame = drop_invalid_rows(dungeon_frame)
    dungeon_frame = get_percentages(dungeon_frame)

    # Escolher um usuário
    user_id = dungeon_frame['PlayerId'].iloc[0]

    # Recomendar atributos
    recommendations = recommend_attributes_hybrid(user_id, dungeon_frame, method)

    return recommendations

def collaborative_filtering(_df, method, teste):

    # ------------- VISUALIZAÇÃO DOS DADOS ------------
    print("--------------------1----------------------------")
    print(_df.shape)
    print(_df.head())
    print("-"*80)

    print("\nFrequência no dungeon_frame final:")
    print(_df['PreFormAnswers'].value_counts())

    # Número de usuários únicos pelo PlayerId
    num_unicos = _df['PlayerId'].nunique()
    print(f"\nNúmero de usuários únicos: {num_unicos}")

    # ------------- NORMALIZAÇÃO DOS DADOS ------------#
    print("--------------------2----------------------------")
    collab_df = build_collab_df(_df, method)
    test_df = build_collab_df(teste, method)

    print("\nMatriz para filtragem 10 :")
    print(test_df.head())

    print("\nMatriz para filtragem 1 :")
    print(collab_df.head())

    # ------------- CÁLCULOS ------------#
    print("--------------------3----------------------------")

    # Create user-item matrix
    user_item_matrix = collab_df.pivot_table(index='User', columns='Item', values='Rating', fill_value=0)
    test_matrix = test_df.pivot_table(index='User', columns='Item', values='Rating', fill_value=0)
    print(test_matrix.head())

    # 🔥 COLOCA AQUI ↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓

    n_users = user_item_matrix.shape[0]
    n_items = user_item_matrix.shape[1]

    total_cells = n_users * n_items
    non_zero = (user_item_matrix > 0).sum().sum()

    sparsity = 1 - (non_zero / total_cells)

    print("\n===== SPARSITY =====")
    print("Users:", n_users)
    print("Items:", n_items)
    print("Total cells:", total_cells)
    print("Non-zero interactions:", non_zero)
    print("Sparsity:", sparsity)

    # ------------- VERIFICAÇÃO DOS DADOS ------------#
    print("--------------------4----------------------------")
    analyze_user_item_matrix(user_item_matrix)

    # ---------------- SIMILIARIDADE
    print("--------------------5----------------------------")
    if(method % 2 == 0):
      item_similarity_df, user_similarity_df = similiarity_pearson(user_item_matrix)
    else:
      item_similarity_df, user_similarity_df = similiarity_cosine(user_item_matrix)

    # ------------- RECOMENDAR ITEM POR ITEM ------------#
    # esta comentado, pq nao fareirpos oturos casos
    '''if(method >=1 and method <=4):
      # Example recommendation
      similar_items = recommend_similar_items('A=4, C=3, I=2, M=1', item_similarity_df)
      print("\nItems similar to A=4, C=3, I=2, M=1:", similar_items)'''

    # Recomendar todos os itens por similarida itens
    print("--------------------6----------------------------")
    '''all_recommendations = {}

    for item in user_item_matrix.columns:
        all_recommendations[item] = recommend_similar_items(item, item_similarity_df)

    print(all_recommendations)

    # ------------- SIMILIARIDA USUARIO POR USUARIO ------------#
    print("--------------------7----------------------------")

    # print abaixo é pra ENTENDNEO O RESULTADO ACIMA
    print_user_ratings(user_item_matrix, user_similarity_df)

    # ------------- testando um usuario com mais de um nivel ------------#
    print("--------------------8----------------------------")
    analyze_user_activity(user_item_matrix)

    # Recomendar para u1 baseado em u2
    print("--------------------9----------------------------")
    recommend_from_other_user(user_item_matrix.index[0], user_item_matrix.index[1], user_item_matrix)

    # Recomendar para u2 baseado em u1
    print("--------------------10----------------------------")
    recommend_from_other_user(user_item_matrix.index[1], user_item_matrix.index[0], user_item_matrix)

    # Escolher usuário
    print("--------------------13----------------------------")
    recommend_user_based_explain(user_item_matrix.index[0], user_item_matrix, user_similarity_df, top_n=5)'''

    # 🔥 MULTI-VIEW RECOMMENDATION (NOVO)
    print("--------------------14----------------------------")
    if method == 13:
        print("\n\n=========== MULTI VIEW ===========")
        multi_view_recommendation(_df, _df['PlayerId'].iloc[0])
    elif (method >= 14):
        print("\n=========== HYBRID RECOMMENDATION ===========")
        recommend_hybrid(user_item_matrix.index[0], user_item_matrix, item_similarity_df, user_similarity_df)


    # ---------------------------------------------
    # PREDIÇÃO APENAS NO TEST SET
    # ---------------------------------------------

    # 1. Garantir interseção estrutural
    print("TEST MATRIX SHAPE BEFORE:", test_matrix.shape)
    print("TRAIN MATRIX SHAPE:", user_item_matrix.shape)

    common_users = set(collab_df["User"]).intersection(test_df["User"])
    common_items = set(collab_df["Item"]).intersection(test_df["Item"])

    print("\nANTES DO FILTRO:")
    print("Test size:", len(test_df))
    print("Common users:", len(common_users))
    print("Common items:", len(common_items))

    test_df = test_df[
        test_df["User"].isin(common_users) &
        test_df["Item"].isin(common_items)
    ]

    print("\nDEPOIS DO FILTRO:")
    print("Test size:", len(test_df))

    if len(test_df) == 0:
        print("\n TEST DF VIROU VAZIO → pulando métricas")
        return


    print("TEST MATRIX SHAPE AFTER:", test_matrix.shape)


    all_items = user_item_matrix.columns

    recommendations_user = []

    for user in test_df["User"].unique():

        for item in all_items:

            score = predict_user_based(
                user,
                item,
                user_item_matrix,
                user_similarity_df
            )

            recommendations_user.append({
                "User": user,
                "Item": item,
                "Score": score
            })
    recommendations_item = []

    for user in test_df["User"].unique():

        for item in all_items:

            score = predict_item_based(
                user,
                item,
                user_item_matrix,
                item_similarity_df
            )

            recommendations_item.append({
                "User": user,
                "Item": item,
                "Score": score
            })
    rec_user_df = pd.DataFrame(recommendations_user)

    rec_user_df = rec_user_df.sort_values(
        ["User", "Score"],
        ascending=[True, False]
    )

    rec_item_df = pd.DataFrame(recommendations_item)

    rec_item_df = rec_item_df.sort_values(
        ["User", "Score"],
        ascending=[True, False]
    )

    predictions = []
    predictions_item = []

    for _, row in test_df.iterrows():

        user = row["User"]
        item = row["Item"]
        true_rating = row["Rating"]

        # ---------- USER-BASED ----------
        pred_user = predict_user_based(
            user,
            item,
            user_item_matrix,
            user_similarity_df
        )

        # ---------- ITEM-BASED ----------
        pred_item = predict_item_based(
            user,
            item,
            user_item_matrix,
            item_similarity_df
        )

        predictions.append({
            "User": user,
            "Item": item,
            "Rating": true_rating,
            "Prediction": pred_user
        })

        predictions_item.append({
            "User": user,
            "Item": item,
            "Rating": true_rating,
            "Prediction": pred_item
        })


    print("\n===== ITEM-BASED =====")

    pred_df = pd.DataFrame(predictions)
    pred_item_df = pd.DataFrame(predictions_item)

    print("\n===== DEBUG INTERSEÇÃO =====")
    print("Itens únicos preditos:", pred_df['Item'].nunique())
    print("Itens únicos no teste:", test_df['Item'].nunique())

    intersection = set(pred_df['Item']) & set(test_df['Item'])
    print("Interseção de itens:", len(intersection))
    print("Itens em comum:", intersection)

    print("\n===== DEBUG USUÁRIOS =====")
    print(test_df.groupby("User")["Item"].count())

    # remover NaN
    pred_df = pred_df.dropna()
    pred_item_df = pred_item_df.dropna()

    rmse = np.sqrt(mean_squared_error(pred_df["Rating"], pred_df["Prediction"]))
    mae  = mean_absolute_error(pred_df["Rating"], pred_df["Prediction"])
    print("Predições válidas:", len(pred_df))
    print("RMSE:", rmse)
    print("MAE:", mae)
    rmse_item = np.sqrt(mean_squared_error(pred_item_df["Rating"], pred_item_df["Prediction"]))
    mae_item  = mean_absolute_error(pred_item_df["Rating"], pred_item_df["Prediction"])
    print("RMSE:", rmse_item)
    print("MAE:", mae_item)
    precision = precision_at_k(pred_df, test_df, k=5, threshold=1.0)
    recall = recall_at_k(pred_df, test_df, k=5, threshold=1.0)

    precision_user = precision_at_k(rec_user_df, test_df, k=8, threshold=0.0)
    recall_user    = recall_at_k(rec_user_df, test_df, k=8, threshold=0.0)
    precision_item = precision_at_k(rec_item_df, test_df, k=8, threshold=0.0)
    recall_item    = recall_at_k(rec_item_df, test_df, k=8, threshold=0.0)

    print("Precision@5:", precision_user)
    print("Recall@5:", recall_user)
    print("Precision@5:", precision_item)
    print("Recall@5:", recall_item)

    hr_user = hit_rate_at_k(rec_user_df, test_df, k=8)
    mrr_user = mrr(rec_user_df, test_df)

    hr_item = hit_rate_at_k(rec_item_df, test_df, k=8)
    mrr_item = mrr(rec_item_df, test_df)

    print("HitRate@5:", hr_user)
    print("MRR:", mrr_user)
    print("HitRate@5:", hr_item)
    print("MRR:", mrr_item)


    return {
        'RMSE': rmse,
        'MAE': mae,
        #'Precision@5': precision,
        #'Recall@5': recall,
        'RMSE_item': rmse_item,
        'MAE_item': mae_item,
        #'Precision@5_item': precision_item,
        #'Recall@5_item': recall_item
    }


def leave_one_out_split(df, user_col='PlayerId'):
    train_list, test_list = [], []
    for u, grp in df.groupby(user_col):
        if len(grp)<2:
            train_list.append(grp); continue
        test_list.append(grp.sample(1, random_state=42))
        train_list.append(grp.drop(test_list[-1].index))
    return pd.concat(train_list), pd.concat(test_list)

def pca_method_advisor(df, target_column='Rating'):
    """
    Usa PCA para analisar a estrutura dos dados e recomendar
    qual método de recomendação usar
    """
    from sklearn.decomposition import PCA
    from sklearn.preprocessing import StandardScaler

    # Selecionar características relevantes para decisão
    decision_features = [
        'UniqueRoomsEntered', 'TotalRooms',  # Estrutura
        'TotalEnemies', 'EnemiesKilled',     # Combate
        'CompletedCreativityQuests',          # Comportamento
        'TotalLostHealth', 'MaxCombo'         # Performance
    ]

    available_features = [f for f in decision_features if f in df.columns]

    # Agrupar por método/usuário
    feature_matrix = df[available_features].fillna(0)

    # Padronizar
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(feature_matrix)

    # PCA para entender estrutura
    pca = PCA(n_components=2)  # 2D para visualização
    X_pca = pca.fit_transform(X_scaled)

    analysis = {
        'explained_variance': pca.explained_variance_ratio_,
        'feature_importance': dict(zip(available_features, pca.components_[0]))
    }

    print("\n=== ANÁLISE PCA - RECOMENDAÇÃO DE MÉTODO ===")
    print(f"Variância PC1: {pca.explained_variance_ratio_[0]:.2%}")
    print(f"Variância PC2: {pca.explained_variance_ratio_[1]:.2%}")

    # Critérios de decisão baseados no PCA
    if pca.explained_variance_ratio_[0] > 0.6:
        print("\n📊 Dados têm ESTRUTURA FORTE (PC1 domina)")
        print("   → Recomendado: Métodos 5-8 (LevelName)")
        print("   → Motivo: Dados se organizam claramente")
        recommendation = 5
    elif pca.explained_variance_ratio_[0] < 0.3:
        print("\n📊 Dados são ESPARSOS (muita variância distribuída)")
        print("   → Recomendado: PCA + métodos 1-4 (Perfis)")
        print("   → Motivo: Muitos padrões fracos")
        recommendation = 1
    else:
        print("\n📊 Dados têm ESTRUTURA MODERADA")
        print("   → Recomendado: Métodos Híbridos (14-19)")
        print("   → Motivo: Combinação de padrões fortes e fracos")
        recommendation = 14

    # Análise de cardinalidade via PCA
    unique_combos = df.groupby(['LevelName', 'PlayerId']).size().shape[0]
    total_rows = len(df)
    sparsity_estimate = 1 - (unique_combos / total_rows) if total_rows > 0 else 1

    print(f"\n💰 Esparsidade estimada: {sparsity_estimate:.2%}")

    if sparsity_estimate > 0.95:
        print("   → Dados MUITO esparsos → Use User-based (métodos 3,4,7,8)")
    elif sparsity_estimate > 0.8:
        print("   → Dados esparsos → Use Item-based (métodos 1,2,5,6)")
    else:
        print("   → Dados densos → Qualquer método funciona")

    return recommendation

def pca_granularity_advisor(df):
    """
    PCA ajuda a decidir se usa LevelName, Perfil, ou Estrutura
    """
    from sklearn.decomposition import PCA

    # Testar diferentes "itens"
    candidates = {
        'LevelName': df['LevelName'].nunique() / len(df),
        'PreFormAnswers': df['PreFormAnswers'].nunique() / len(df),
        'Estrutura': (df['TotalRooms'].astype(str) +
                      df['TotalKeys'].astype(str)).nunique() / len(df)
    }

    # Usar PCA para ver estrutura latente
    features = ['TotalRooms', 'TotalKeys', 'TotalEnemies', 'TotalTreasure']
    available = [f for f in features if f in df.columns]

    X = df[available].fillna(0)
    pca = PCA(n_components=2)
    X_pca = pca.fit_transform(X)

    # Análise de separabilidade
    level_groups = df.groupby('LevelName').size()
    profile_groups = df.groupby('PreFormAnswers').size()

    print("\n=== PCA - ANÁLISE DE GRANULARIDADE ===")
    print(f"Cardinalidade LevelName: {candidates['LevelName']:.2%}")
    print(f"Cardinalidade Perfil: {candidates['PreFormAnswers']:.2%}")
    print(f"Cardinalidade Estrutura: {candidates['Estrutura']:.2%}")
    print(f"Variância explicada: {pca.explained_variance_ratio_.sum():.2%}")

    # Decisão
    if candidates['LevelName'] < 0.05 and len(level_groups) > 5:
        print("\n✅ Recomendado: Usar LevelName (métodos 5-8)")
        return 'LevelName'
    elif candidates['PreFormAnswers'] < 0.1:
        print("\n✅ Recomendado: Usar Perfil (métodos 1-4)")
        return 'PreFormAnswers'
    else:
        print("\n✅ Recomendado: Usar estrutura discretizada (métodos 9-12)")
        return 'Structure'
def pca_data_filter(df, variance_threshold=0.85):
    """
    Usa PCA para filtrar quais colunas/features são importantes
    """
    from sklearn.decomposition import PCA
    from sklearn.preprocessing import StandardScaler

    # Todas colunas numéricas
    numeric_cols = df.select_dtypes(include=['int64', 'float64']).columns
    exclude = ['PlayerId', 'TotalAttempts']
    feature_cols = [c for c in numeric_cols if c not in exclude]

    X = df[feature_cols].fillna(0)
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    # PCA com todos componentes
    pca = PCA(n_components=len(feature_cols))
    pca.fit(X_scaled)

    # Encontrar quantos componentes atingem variance_threshold
    cumsum = np.cumsum(pca.explained_variance_ratio_)
    n_components = np.argmax(cumsum >= variance_threshold) + 1

    # Identificar features mais importantes
    importance = np.abs(pca.components_[:n_components]).sum(axis=0)
    feature_importance = pd.DataFrame({
        'feature': feature_cols,
        'importance': importance
    }).sort_values('importance', ascending=False)

    print("\n=== PCA FILTER - FEATURES IMPORTANTES ===")
    print(f"Total features: {len(feature_cols)}")
    print(f"Features que explicam {variance_threshold:.0%} da variância: {n_components}")
    print(f"\nTop 10 features mais importantes:")
    print(feature_importance.head(10))

    # Recomendar filtrar features com baixa importância
    threshold = feature_importance['importance'].median()
    important_features = feature_importance[
        feature_importance['importance'] > threshold
    ]['feature'].tolist()

    print(f"\n📊 Features a manter: {len(important_features)}")
    print(f"   Exemplo: {important_features[:5]}")

    return important_features

# Uso
def classify_with_pca_filter(_df, method):
    # Primeiro, descobre features importantes
    important_features = pca_data_filter(_df, variance_threshold=0.85)

    # Filtra o dataframe
    keep_cols = ['PlayerId', 'LevelName', 'PreFormAnswers'] + important_features
    available_cols = [c for c in keep_cols if c in _df.columns]
    _df_filtered = _df[available_cols].copy()

    print(f"\n📊 Dados filtrados: {len(available_cols)} colunas")

    # Continua com o método original
    classify(_df_filtered, method)

# Uso no seu código
def main_with_advisor(method=None):
    initialize_data_frame()
    dungeon_frame = get_dungeon_data_frame()
    user_frame = get_user_data_frame()
    dungeon_frame = pd.merge(dungeon_frame, user_frame[['PlayerId', 'PreFormAnswers']], on=['PlayerId'], how='left')
    dungeon_frame = drop_invalid_rows(dungeon_frame)
    dungeon_frame = drop_unused_columns(dungeon_frame)
    dungeon_frame = get_percentages(dungeon_frame)

    # Se método não foi especificado, usa PCA para recomendar
    if method is None:
        method = pca_method_advisor(dungeon_frame)
        print(f"\n🎯 PCA recomendou usar método {method}")

    classify(dungeon_frame, method)

def analyze_split_cardinality(_df, method=1):
    """
    Analisa a cardinalidade do item gerado pelo método
    ANTES de fazer o split
    """
    collab_df = build_collab_df(_df, method)

    print(f"\n{'='*60}")
    print(f"MÉTODO {method} - Análise de Cardinalidade do Item")
    print(f"{'='*60}")

    # Análise básica
    n_total = len(collab_df)
    n_unique_items = collab_df['Item'].nunique()
    cardinality = n_unique_items / n_total

    print(f"Total de interações (usuário-item): {n_total}")
    print(f"Itens únicos gerados: {n_unique_items}")
    print(f"Cardinalidade: {cardinality:.4f} ({cardinality*100:.2f}%)")

    # Distribuição dos itens
    item_counts = collab_df['Item'].value_counts()
    print(f"\nMédia de ocorrências por item: {n_total / n_unique_items:.2f}")
    print(f"Mediana de ocorrências: {item_counts.median():.2f}")
    print(f"Desvio padrão: {item_counts.std():.2f}")

    # Itens que se repetem
    repeated_items = item_counts[item_counts > 1]
    print(f"\nItens com repetição (>1): {len(repeated_items)} / {n_unique_items}")
    print(f"Porcentagem de itens repetidos: {len(repeated_items)/n_unique_items*100:.2f}%")

    # Itens singleton (aparecem 1 vez)
    singleton_items = item_counts[item_counts == 1]
    print(f"Itens que aparecem APENAS 1 vez: {len(singleton_items)} / {n_unique_items}")
    print(f"Porcentagem de singletons: {len(singleton_items)/n_unique_items*100:.2f}%")

    # Verificar se vai funcionar
    if len(repeated_items) == 0:
        print("\n🚨 PROBLEMA CRÍTICO: NENHUM item se repete!")
        print("   → O split vai falhar porque não haverá interseção treino/teste")
    elif len(repeated_items) < 10:
        print("\n⚠️ ALERTA: Muito poucos itens se repetem!")
        print("   → O split pode ter interseção muito pequena")
    else:
        print("\n✅ OK: Existem itens repetidos suficientes")
        print(f"   → {len(repeated_items)} itens podem ser compartilhados entre treino/teste")

    # Mostrar exemplos
    print(f"\nExemplos de itens que se repetem:")
    for item, count in item_counts.head(5).items():
        print(f"  '{item}': {count} ocorrências")

    return {
        'method': method,
        'n_interactions': n_total,
        'n_unique_items': n_unique_items,
        'cardinality': cardinality,
        'n_repeated_items': len(repeated_items),
        'n_singletons': len(singleton_items),
        'will_work': len(repeated_items) > 0
    }


def analyze_pca_item(df):
    """
    Analisa se PCA vai ajudar na cardinalidade
    """
    structural_features = [
        'TotalRooms', 'TotalKeys', 'TotalLocks',
        'TotalEnemies', 'TotalTreasure', 'TotalNpcs'
    ]

    available_features = [f for f in structural_features if f in df.columns]
    X = df[available_features].fillna(0)

    # PCA com todos componentes para ver variância
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    pca = PCA(n_components=len(available_features))
    pca.fit(X_scaled)

    print("\n=== ANÁLISE PCA ===")
    print(f"Features originais: {len(available_features)}")
    print(f"Variância acumulada:")

    cumulative_var = 0
    for i, var in enumerate(pca.explained_variance_ratio_):
        cumulative_var += var
        print(f"  PC{i+1}: {var:.2%} (acumulado: {cumulative_var:.2%})")
        if cumulative_var >= 0.85:
            print(f"  → {i+1} componentes explicam 85% da variância")
            break


def classify(_df, method):
    _df['PreFormAnswers'] = _df['PreFormAnswers'].apply(lambda x: 'A=' + str(x[0]) + ", C=" + str(x[1]) + ", I=" + str(x[2]) +
                                                                  ", M=" + str(x[3]))
    _df = split_preform_answers(_df)
    n = _df.shape[0]

    print("_df shape:", _df.shape)

    # 🔴 FILTRO ANTES DO SPLIT (ESSENCIAL)
    counts = _df.groupby("PlayerId").size()
    valid_users = counts[counts >= 2].index
    _df = _df[_df["PlayerId"].isin(valid_users)]

    # 🔵 SPLIT
    train_df, test_df = leave_one_out_split(_df)


    numeric_cols = train_df.select_dtypes(include=["float64", "int64"]).columns

    # REMOVER colunas estruturais
    cols_to_exclude = ["PlayerID", "PreFormAnswers", "LevelName"]
    numeric_cols = [c for c in numeric_cols if c not in cols_to_exclude]

    scaler = MinMaxScaler()

    train_df[numeric_cols] = scaler.fit_transform(train_df[numeric_cols])
    test_df[numeric_cols] = scaler.transform(test_df[numeric_cols])



    _dfnoise = augment_data_random(train_df, ['PreFormAnswers', 'A', 'C', 'I', 'M'], n)
    _dfscale = augment_dataframe_with_scaling(train_df, ['PreFormAnswers', 'A', 'C', 'I', 'M'], n, 1.05)
    _dfnoise2 = augment_data_random(train_df, ['PreFormAnswers', 'A', 'C', 'I', 'M'], 2*n)
    _dfscale2 = augment_dataframe_with_scaling(train_df, ['PreFormAnswers', 'A', 'C', 'I', 'M'], n, 1.05)
    _dfscale2 = augment_dataframe_with_scaling(train_df, ['PreFormAnswers', 'A', 'C', 'I', 'M'], n, 0.95)
    _dfmixed = augment_data_random(train_df, ['PreFormAnswers', 'A', 'C', 'I', 'M'], n)
    _dfmixed = pd.concat([_dfmixed, augment_dataframe_with_scaling(train_df, ['PreFormAnswers', 'A', 'C', 'I', 'M'], n, 1.05)], ignore_index=True)
    _dfmixed10 = augment_data_random(train_df, ['PreFormAnswers', 'A', 'C', 'I', 'M'], 6*n)
    _dfmixed10 = pd.concat([_dfmixed10, augment_dataframe_with_scaling(train_df, ['PreFormAnswers', 'A', 'C', 'I', 'M'], n, 1.05)], ignore_index=True)
    _dfmixed10 = pd.concat([_dfmixed10, augment_dataframe_with_scaling(train_df, ['PreFormAnswers', 'A', 'C', 'I', 'M'], n, 0.95)], ignore_index=True)
    _dfmixed10 = pd.concat([_dfmixed10, augment_dataframe_with_scaling(train_df, ['PreFormAnswers', 'A', 'C', 'I', 'M'], n, 1.10)], ignore_index=True)
    _dfmixed10 = pd.concat([_dfmixed10, augment_dataframe_with_scaling(train_df, ['PreFormAnswers', 'A', 'C', 'I', 'M'], n, 0.9)], ignore_index=True)

    dataframes = [#_df, _dfnoise, _dfscale, _dfnoise2, _dfscale2, _dfmixed,
                  _dfmixed10]
    experiment_names = [#'Baseline', '1xN', '1xS-1_05', '2xN', '2xS-1_05+0_95', '1xS-1_05+1xN',
                        '6xN+4xS-1_05+0_95+1_1+0_9']

    strategies = [#'Original', '1xNoise', '1xScale', '2xN', '2xScale', '1xN+1xS',
                  '6xN+4xS']

    # Quantos níveis diferentes cada usuário jogou
    levels_per_user = _df.groupby("PlayerId")["LevelName"].nunique()

    # Filtrar apenas quem jogou mais de 1 nível
    multi_level_users = levels_per_user[levels_per_user > 1]

    print("Usuários que jogaram mais de um nível:")
    print(multi_level_users)

    print("\nDetalhamento:\n")

    '''for user in multi_level_users.index:
        levels = _df[_df["PlayerId"] == user]["LevelName"].unique()
        print(f"Usuário {user} jogou:")
        print(list(levels))
        print("-" * 50)'''

    #pca_method_advisor(_df)
    #pca_granularity_advisor(_df)
    #pca_data_filter(_df)
    collaborative_filtering(_dfmixed10, method, test_df)



def reset_results():
    Results.accuracy = pd.DataFrame()
    Results.precision = pd.DataFrame()
    Results.recall = pd.DataFrame()
    Results.f1 = pd.DataFrame()
    Results.kappa = pd.DataFrame()
    Results.train_accuracy = pd.DataFrame()
    Results.train_precision = pd.DataFrame()
    Results.train_recall = pd.DataFrame()
    Results.train_f1 = pd.DataFrame()
    Results.train_kappa = pd.DataFrame()
    Results.cv_mean = pd.DataFrame()
    Results.cv_stddev = pd.DataFrame()
    Results.train_error_rate = pd.DataFrame()
    Results.gap = pd.DataFrame()
    Results.error_rate = pd.DataFrame()
    Results.train_time = pd.DataFrame()
    Results.infer_time = pd.DataFrame()

def main(method):
    initialize_data_frame()
    dungeon_frame = get_dungeon_data_frame()
    user_frame = get_user_data_frame()
    dungeon_frame = pd.merge(dungeon_frame, user_frame[['PlayerId', 'PreFormAnswers']], on=['PlayerId'], how='left')
    dungeon_frame = drop_invalid_rows(dungeon_frame)
    #dungeon_frame = drop_unused_columns(dungeon_frame)
    dungeon_frame = get_percentages(dungeon_frame)
    #dungeon_frame = drop_processed_columns(dungeon_frame)
    classify(dungeon_frame, method)
    '''evaluate_collaborative_filtering(
        dungeon_frame,
        method=1,
        top_k=10
    )'''


def teste():
    initialize_data_frame()

    dungeon_frame = get_dungeon_data_frame()
    user_frame = get_user_data_frame()

    # 🔥 CONVERTE PARA TUPLA AQUI
    profiles = user_frame['PreFormAnswers'].apply(tuple)

    print("Combinações únicas:")
    print(sorted(profiles.unique()))

    print("\nQuantidade de combinações únicas:")
    print(profiles.nunique())

    print("\nFrequência de cada combinação:")
    print(profiles.value_counts())

    dungeon_frame = pd.merge(
        dungeon_frame,
        user_frame[['PlayerId', 'PreFormAnswers']],
        on=['PlayerId'],
        how='left'
    )

    print("\nFrequência no dungeon_frame final:")
    print(dungeon_frame['PreFormAnswers'].value_counts())

    # Listar todas as colunas
    colunas = dungeon_frame.columns.tolist()

    print(f"\nTotal de colunas: {len(colunas)}\n")

    # Exibir em ordem alfabética numerada
    for i, col in enumerate(sorted(colunas), 1):
        print(f"{i:3d}. {col}")

In [38]:
main(5) # 1 E 5 FOI, VER PQ 9 NAO FOI

/tmp/ipykernel_1100/1505721220.py:223: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  _data_frame.replace([np.inf, -np.inf], 0, inplace=True)


_df shape: (119, 67)
Usuários que jogaram mais de um nível:
PlayerId
53627948       4
78194049       3
81972292      11
121412094     12
282268519      4
1093542367    12
1153731241     4
1776427228     2
2079201442     4
2079942857    12
2096844718     3
Name: LevelName, dtype: int64

Detalhamento:

--------------------1----------------------------
(1710, 67)
                      PostFormAnswers  TotalEnemies  TotalMasteryQuests  MaxCombo  TotalExploreQuests  TotalNpcs  CompletedReportQuests  CompletedListenQuests  TotalDeaths  CompletedAchievementQuests  TotalGiveQuests  TotalAchievementQuests  TotalKeys  PlayerId  TotalLocks  KeysCollected  TotalExchangeQuests  TotalLostHealth  CompletedExploreQuests  CompletedExchangeQuests  TotalGoToQuests  RoomsEntered  CompletedDamageQuests  TotalAttempts  TotalCreativityQuests  TotalGatherQuests  NpcsInteracted  CompletedMasteryQuests                                       InputProfile  CompletedGiveQuests WeaponName  TotalImmersionQuests      

/usr/local/lib/python3.12/dist-packages/numpy/_core/fromnumeric.py:3596: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/usr/local/lib/python3.12/dist-packages/numpy/_core/_methods.py:138: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


In [ ]:
# eu consigo recomendar niveis, com base em itens
# fazer recoencado de niveis com base granualr, cada tabela
# dificuldade - eu precisaria ver o nivel simialr a aquela dficiudalde e ver a avalicao da pessoa, se ela gsotou

#recomendar level e explica - recomendado pq tem x salsa, filtrar
#criar item composto ex: D: 10, C: 20, I: 4 - DIFICUA, CHAVE, INIMIGO... - FAZRE UM TIPO DE NOME PARA CASOS - AI USAR COLUNA
#ANTES DE IR RECOMENDADO PERIMEIRO JUNTAR


In [ ]:
# dificuldade de inimigos
# quests
# matar inimigo
# explorar dungeon
#Dugeon com esses dados: 20 salas, 4 chaves-fechaduras, 1.5 de linearidade - CALCULA TUDO SEARADO E JUTNA, TALVEZ



## Atributos Estruturais
1. **TotalRooms** - número de salas
2. **TotalKeys** - número de chaves
3. **TotalLocks** - número de fechaduras
4. **UniqueRoomsEntered** - salas únicas visitadas
5. **RoomsEntered** - total de salas entradas (com revisitas)

## Atributos de Combate
6. **TotalEnemies** - número total de inimigos
7. **EnemiesKilled** - inimigos mortos
8. **MaxCombo** - maior combo
9. **TotalLostHealth** - vida perdida
10. **TotalDeaths** - número de mortes

## Atributos de Interação
11. **TotalNpcs** - número de NPCs
12. **NpcsInteracted** - NPCs com quem interagiu

## Atributos de Progresso
13. **TotalTreasure** - tesouros disponíveis
14. **TreasuresCollected** - tesouros coletados
15. **TimeToFinish** - tempo para finalizar

## Atributos de Missões (Quests)
16. **TotalAchievementQuests** - total de quests de conquista
17. **TotalCreativityQuests** - total de quests de criatividade
18. **TotalDamageQuests** - total de quests de dano
19. **TotalExploreQuests** - total de quests de exploração
20. **TotalGatherQuests** - total de quests de coleta
21. **TotalGiveQuests** - total de quests de entrega
22. **TotalGoToQuests** - total de quests de "ir até"
23. **TotalImmersionQuests** - total de quests de imersão
24. **TotalKillQuests** - total de quests de matar
25. **TotalListenQuests** - total de quests de ouvir
26. **TotalMasteryQuests** - total de quests de mestria
27. **TotalReadQuests** - total de quests de ler
28. **TotalReportQuests** - total de quests de relatório

## Métricas Calculadas
29. **LockUsage** - uso de fechaduras (LocksOpened/TotalLocks)
30. **KeyCollection** - coleta de chaves (KeysCollected/TotalKeys)
31. **MapCompletion** - exploração (UniqueRoomsEntered/TotalRooms)
32. **RoomRevisitRate** - taxa de revisita
33. **EnemyKillRate** - taxa de abate (EnemiesKilled/TotalEnemies)
34. **Success Rate** - taxa de sucesso (Wins/Attempts)



#TODO
- 1º Analises -> precsiao,recall, rmse, mae
- 2º Fazer a tabela para cada atribulo - inmigos, quest, difuclda, prefenais (maar, explora...) (deixre indicado para cada um) um multiclasse
- 3º Normalizar os dados (1 a 0)
- 4º PCA (usar todos os dados e depois usar o pca para ver quais sao inuties, assim tirar eles)
- 5º fazer as analises como dito no artigo (eu tenho que faze-las
- 6º Organizar o código mais ainda

#add no artigo
- poderia usar o perfil para melhora a recomendação
- recomendaco a aprtir de todos usuarios? ex.: um grupo aprova isso e vc ta no grupo, entao vc vai gostar



